第一节，openai工具调用

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
import openai
openai.api_key = os.getenv("OPENAI_API_KEY")

In [3]:
import json

def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""
    weather_info = {
        "location": location,
        "unit": unit,
        "temperature": 75,
        "description": ["sunny", "windy"]
    }
    return json.dumps(weather_info)


In [4]:
# define a function
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object", 
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA"
                        },
                    "unit": {"type": "string", "enum": ["fahrenheit", "celsius"]}
            },
            "required": ["location"]
        }
    }
]


In [5]:
message = [
    {
        "role": "user", 
        "content": "What is the weather in San Francisco?"
    }
]


In [6]:
from openai import OpenAI
# api_key 和 base_url 在创建客户端时传入，不能传给 create()
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.longcat.chat/openai"
)
tools = [{"type": "function", "function": f} for f in functions]
response = client.chat.completions.create(
    model="LongCat-Flash-Chat",
    messages=message,
    tools=tools,
    tool_choice="auto"
)

In [7]:
# 按行分开、易读地打印（转成字典再格式化为 JSON）
import json
print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

{
  "id": "faac07a8672e47cf8dad58870f89daea",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_1b095e67dbe546299d5301cb",
            "function": {
              "arguments": "{\"location\": \"San Francisco, CA\", \"unit\": \"fahrenheit\"}",
              "name": "get_current_weather"
            },
            "type": "function",
            "index": null
          }
        ]
      },
      "matched_stop": null
    }
  ],
  "created": 1773740038,
  "model": "longcat-flash-chatai-api",
  "object": "chat.completion",
  "service_tier": null,
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 34,
    "prompt_tokens": 265,
    "total_tokens": 299,
    "completion_tokens_details": null,

In [8]:
# v1.0+ 的 response 是对象，用属性访问：.choices[0].message
response_message = response.choices[0].message

In [9]:
print(json.dumps(response_message.model_dump(), indent=2, ensure_ascii=False))

{
  "content": null,
  "refusal": null,
  "role": "assistant",
  "annotations": null,
  "audio": null,
  "function_call": null,
  "tool_calls": [
    {
      "id": "call_1b095e67dbe546299d5301cb",
      "function": {
        "arguments": "{\"location\": \"San Francisco, CA\", \"unit\": \"fahrenheit\"}",
        "name": "get_current_weather"
      },
      "type": "function",
      "index": null
    }
  ]
}


In [10]:
response_message.content

In [11]:
print(json.dumps([tc.model_dump()
      for tc in response_message.tool_calls], indent=2))

[
  {
    "id": "call_1b095e67dbe546299d5301cb",
    "function": {
      "arguments": "{\"location\": \"San Francisco, CA\", \"unit\": \"fahrenheit\"}",
      "name": "get_current_weather"
    },
    "type": "function",
    "index": null
  }
]


In [12]:
# tool_calls 是列表，取第一个；参数名是 arguments（复数）
json.loads(response_message.tool_calls[0].function.arguments)

{'location': 'San Francisco, CA', 'unit': 'fahrenheit'}

In [13]:
args = json.loads(response_message.tool_calls[0].function.arguments)

In [14]:
get_current_weather(args)

'{"location": {"location": "San Francisco, CA", "unit": "fahrenheit"}, "unit": "fahrenheit", "temperature": 75, "description": ["sunny", "windy"]}'